## Extract data

In [442]:
import pandas as pd
import numpy as np

In [443]:
import os
print(os.getcwd())

/Users/Kirill/Documents/GitHub/Data-Warehouse


In [444]:
# # Ensure the required library is installed 
# (Can we do it (openpyxl)? Will it be a problem?)
# %pip install openpyxl

# Read the Excel file
file_path = "bitre_fatalities_dec2024.xlsx"
df = pd.read_excel(file_path, sheet_name="BITRE_Fatality", skiprows=4) # Can be improved
print(df.head())


   Crash ID State  Month  Year Dayweek      Time Crash Type Bus Involvement  \
0  20241115   NSW     12  2024  Friday  04:00:00     Single              No   
1  20241125   NSW     12  2024  Friday  06:15:00     Single              No   
2  20246013   Tas     12  2024  Friday  09:43:00   Multiple              No   
3  20241002   NSW     12  2024  Friday  10:35:00   Multiple              No   
4  20242261   Vic     12  2024  Friday  11:30:00   Multiple              -9   

  Heavy Rigid Truck Involvement Articulated Truck Involvement  ... Age  \
0                            No                            No  ...  74   
1                            No                            No  ...  19   
2                            No                            No  ...  33   
3                            No                            No  ...  32   
4                            -9                            -9  ...  62   

  National Remoteness Areas                           SA4 Name 2021  \
0  Inner 

In [445]:
# Clean the columnnames
# Remove leading and trailing whitespace, convert to lowercase, and replace spaces with underscores

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.columns

Index(['crash_id', 'state', 'month', 'year', 'dayweek', 'time', 'crash_type',
       'bus_involvement', 'heavy_rigid_truck_involvement',
       'articulated_truck_involvement', 'speed_limit', 'road_user', 'gender',
       'age', 'national_remoteness_areas', 'sa4_name_2021',
       'national_lga_name_2021', 'national_road_type', 'christmas_period',
       'easter_period', 'age_group', 'day_of_week', 'time_of_day'],
      dtype='object')

In [446]:
# Add a serial number for each person killed in the accident
df['victim_number'] = df.groupby('crash_id').cumcount() + 1

# move the victim_number column to the front
cols = df.columns.tolist()
cols.insert(1, cols.pop(cols.index('victim_number')))
df = df[cols]
print(df.head())



   crash_id  victim_number state  month  year dayweek      time crash_type  \
0  20241115              1   NSW     12  2024  Friday  04:00:00     Single   
1  20241125              1   NSW     12  2024  Friday  06:15:00     Single   
2  20246013              1   Tas     12  2024  Friday  09:43:00   Multiple   
3  20241002              1   NSW     12  2024  Friday  10:35:00   Multiple   
4  20242261              1   Vic     12  2024  Friday  11:30:00   Multiple   

  bus_involvement heavy_rigid_truck_involvement  ... age  \
0              No                            No  ...  74   
1              No                            No  ...  19   
2              No                            No  ...  33   
3              No                            No  ...  32   
4              -9                            -9  ...  62   

  national_remoteness_areas                           sa4_name_2021  \
0  Inner Regional Australia                                Riverina   
1  Inner Regional Australia 

In [447]:
# Save the cleaned DataFrame to a new Excel file
# output_file_path = "bitre_fatalities_cleaned.xlsx"
# df.to_excel(output_file_path, index=False)
# print(f"Cleaned data saved to {output_file_path}")


## Data transformation

Data transformation to apply:

1. dayweek: Drop
2. time: Categorise into rush time and usual time:
    Rush hours: 
    Morning Peak:
    Typically between 7 am and 9 am, as commuters head to work or school. 

    Evening Peak:
    Typically between 4 pm and 6 pm, as commuters travel home from work or school. 

    Not holiday, not weekend

Can be improved according to the state, city and so on

3. bus_involvement, heavy_rigid_truck_involvement, articulated_truck_involvement - treat -9 missing values
4. speed_limit: Categorise as follows:
    For all except NT:
        0-40 - low
        41-50 - med
        51-80 - high
        81 - inf - very high
    
    For NT:
        0-40 - low
        41-60 - med
        61-80 - high
        81 - inf - very high

    treat -9 as missing value
5. road_user:
    treat Other/-9, Unknown - as missing value

6. gender:
    treat -9 - as missing value

7. age: drop

8. national_remoteness_areas:
    treat Unknown - as missing value

9. sa4_name_2021:
    treat Unknown, Blank - as missing value

10. national_lga_name_2021:
    treat Unknown, Blank - as missing value

11. national_road_type:
    treat Undetermined - as missing value

12. christmas_period, easter_period:
    transform into is_holiday

13. age_group:
    treat -9 - as missing value

14. day_of_week:
    treat Unknown - as missing value

15. time_of_day:
    treat Unknown - as missing value

In [448]:
# Create new variable 'holiday' base on christmas_period, easter_period. If either is true, then holiday = 1, else 0
df['holiday'] = np.where(
    (df['christmas_period'].fillna(0) == "Yes") | (df['easter_period'].fillna(0) == "Yes"),
    "Yes",
    "No"
)


In [449]:
# convert the 'time' column to datetime format
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.time

# categorize time of day by rush hous:
# For all holiday == "No", day_of_week == "Weekday" set rush: 
# 07:00:00 - 09:00:00 = "Rush"
# 16:00:00 - 18:00:00 = "Rush"
Weekday = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
df['time'] = np.where(
    (df['holiday'] == "No") & (df['dayweek'].isin(Weekday)) & (
        ((df['time'] >= pd.to_datetime("07:00:00", format='%H:%M:%S').time()) & (df['time'] <= pd.to_datetime("09:00:00", format='%H:%M:%S').time())) |
        ((df['time'] >= pd.to_datetime("16:00:00", format='%H:%M:%S').time()) & (df['time'] <= pd.to_datetime("18:00:00", format='%H:%M:%S').time()))
    ),
    "Rush",
    "Not Rush"
)
print(df['time'].unique())


['Not Rush' 'Rush']


In [450]:
# set  NaN values for all 'Other/-9', '-9', 'Unknown', 'Undetermined' in all columns
nan_values = ['Other/-9', '-9', 'Unknown', 'Undetermined', -9]
df.replace(nan_values, np.nan, inplace=True)
print(df.head())

   crash_id  victim_number state  month  year dayweek      time crash_type  \
0  20241115              1   NSW     12  2024  Friday  Not Rush     Single   
1  20241125              1   NSW     12  2024  Friday  Not Rush     Single   
2  20246013              1   Tas     12  2024  Friday  Not Rush   Multiple   
3  20241002              1   NSW     12  2024  Friday  Not Rush   Multiple   
4  20242261              1   Vic     12  2024  Friday  Not Rush   Multiple   

  bus_involvement heavy_rigid_truck_involvement  ...  \
0              No                            No  ...   
1              No                            No  ...   
2              No                            No  ...   
3              No                            No  ...   
4             NaN                           NaN  ...   

  national_remoteness_areas                           sa4_name_2021  \
0  Inner Regional Australia                                Riverina   
1  Inner Regional Australia  Sydney - Baulkham Hills

In [451]:
# check for right data type 
try :
    df['speed_limit'] = df['speed_limit'].astype(float)
except ValueError:
    # If conversion to int fails, print values that cannot be converted
    print("Values that cannot be converted to int:")
    print(df[~df['speed_limit'].apply(lambda x: isinstance(x, int) or pd.isna(x))]['speed_limit'].unique())



Values that cannot be converted to int:
['<40']


<40 means less than 40, which is 'low' speed category
just set this value to 40

In [452]:
# set '<40' speed_limit to 40
df['speed_limit'] = df['speed_limit'].replace('<40', 40)

In [453]:
# speed_limit: Categorise as follows:
#     For all except NT:
#         0-40 - low
#         41-50 - med
#         51-80 - high
#         81 - inf - very high
    
#     For NT:
#         0-40 - low
#         41-60 - med
#         61-80 - high
#         81 - inf - very high

df['speed_limit'] = np.where(
    df['state'] != "NT",
    np.select(
        [
            (df['speed_limit'] > 0) & (df['speed_limit'] <= 40),
            (df['speed_limit'] >= 41) & (df['speed_limit'] <= 50),
            (df['speed_limit'] >= 51) & (df['speed_limit'] <= 80),
            (df['speed_limit'] > 80)
        ],
        ['Low', 'Med', 'High', 'Very High'],
        default=np.nan
    ),
    np.select(
        [
            (df['speed_limit'] > 0) & (df['speed_limit'] <= 40),
            (df['speed_limit'] >= 41) & (df['speed_limit'] <= 60),
            (df['speed_limit'] >= 61) & (df['speed_limit'] <= 80),
            (df['speed_limit'] > 80)
        ],
        ['Low', 'Med', 'High', 'Very High'],
        default=np.nan
    )
)

# change 'nan' value to np.nan
df['speed_limit'] = df['speed_limit'].replace('nan', np.nan)            # КОСТЫЛЬ
# check if there are any NaN values in speed_limit
print(df['speed_limit'].isna().sum())


1485


In [454]:
print(df['speed_limit'].unique())

['Very High' 'High' 'Med' nan 'Low']


In [455]:
# drop dayweek, age, christmas_period and easter_period columns
df.drop(columns=['dayweek', 'age', 'christmas_period', 'easter_period'], inplace=True)
print(df.head())

   crash_id  victim_number state  month  year      time crash_type  \
0  20241115              1   NSW     12  2024  Not Rush     Single   
1  20241125              1   NSW     12  2024  Not Rush     Single   
2  20246013              1   Tas     12  2024  Not Rush   Multiple   
3  20241002              1   NSW     12  2024  Not Rush   Multiple   
4  20242261              1   Vic     12  2024  Not Rush   Multiple   

  bus_involvement heavy_rigid_truck_involvement articulated_truck_involvement  \
0              No                            No                            No   
1              No                            No                            No   
2              No                            No                            No   
3              No                            No                            No   
4             NaN                           NaN                           NaN   

   ...  road_user  gender national_remoteness_areas  \
0  ...     Driver    Male  Inner Regi

In [456]:
# Save the cleaned DataFrame to a new Excel file
output_file_path = "bitre_fatalities_cleaned2.xlsx"
df.to_excel(output_file_path, index=False)
print(f"Cleaned data saved to {output_file_path}")

Cleaned data saved to bitre_fatalities_cleaned2.xlsx


## Feature engineering (merging data)

you are required to utilise at least one of the following datasets: Dwelling Count Data or Population Data. You may choose to incorporate both of these additional datasets if desired.